# Profiling Real Models in PyTorch: From torch.profiler to Nsight

**Red Hat PyTorch Team | IISC Bangalore**

---

| Hardware | Spec |
|----------|------|
| GPU | 8× NVIDIA H200 SXM (Hopper, SM 9.0) |
| HBM | 143 GB HBM3e per GPU |
| PyTorch | 2.14.0.dev (nightly) |
| CUDA | 12.8 |

**Workshop flow (30+15 min):**
- Part 1 — torch.profiler Foundation (10 min): Sections 1–3
- Part 2 — SOTA Kernel Profiling (15 min): Sections 4–8
- Part 3 — Nsight Deep Dive (10 min): Sections 9–10

Every script profiles **real HuggingFace Transformers layers** on H200 — not toy operations.

## Section 0: Setup & Environment Check

In [ ]:
import os
os.environ["PATH"] = "/usr/local/cuda-12.8/bin:" + os.environ.get("PATH", "")

import torch
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16
TRACE_DIR = "../traces"
os.makedirs(TRACE_DIR, exist_ok=True)

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU       : {props.name}")
    print(f"SM        : {props.major}.{props.minor}  |  SMs: {props.multi_processor_count}")
    print(f"VRAM      : {props.total_mem / 1024**3:.1f} GB")
    print(f"PyTorch   : {torch.__version__}")
    print(f"CUDA      : {torch.version.cuda}")
    print(f"\nHopper FP8   : {'Yes' if props.major >= 9 else 'No'}")
    print(f"cuDNN SDPA   : {'Yes' if props.major >= 9 else 'No (needs Hopper)'}")
else:
    print("CUDA not available — most cells will not run")

In [ ]:
# Check SOTA kernel availability
checks = {}

try:
    from flash_attn_interface import flash_attn_func
    checks["FlashAttention-3"] = "available"
except ImportError:
    checks["FlashAttention-3"] = "not installed (build from flash-attention/hopper/)"

try:
    from flash_mla import get_mla_metadata
    checks["FlashMLA"] = "available"
except ImportError:
    checks["FlashMLA"] = "not installed (build from source)"

try:
    from liger_kernel.transformers import apply_liger_kernel_to_llama
    import liger_kernel
    checks["Liger Kernel"] = f"v{liger_kernel.__version__}"
except ImportError:
    checks["Liger Kernel"] = "not installed (pip install liger-kernel)"

try:
    _ = torch.ones(1, device="cuda", dtype=torch.float8_e4m3fn)
    checks["FP8 (float8_e4m3fn)"] = "supported"
except Exception:
    checks["FP8 (float8_e4m3fn)"] = "not supported on this GPU"

for k, v in checks.items():
    print(f"  {k:<25s} : {v}")

In [ ]:
def perfetto_url(trace_path):
    """Print a Perfetto link for viewing Chrome traces."""
    print(f"Trace saved -> {trace_path}")
    print(f"View at     -> https://ui.perfetto.dev/  (drag-and-drop the JSON file)")

---
# Part 1: torch.profiler Foundation
---

## Section 1: Warmup — matmul + add

### What to expect

The simplest possible profiling target: `torch.matmul(a, b) + bias`.

- **Small matrix (64×64):** Launch overhead dominates. The gap *between* kernels is larger than the kernels themselves.
- **Large matrix (4096×4096):** A single cuBLAS GEMM fills the GPU timeline. This is compute-bound.
- **With `torch.compile`:** Both ops fuse into a single Triton kernel — matmul+add in one launch.

In [ ]:
def matmul_add(a, b, bias):
    return torch.matmul(a, b) + bias

# --- Small: 64x64 (overhead-bound) ---
n = 64
a = torch.randn(n, n, device=DEVICE, dtype=DTYPE)
b = torch.randn(n, n, device=DEVICE, dtype=DTYPE)
bias = torch.randn(n, device=DEVICE, dtype=DTYPE)

for _ in range(5):
    matmul_add(a, b, bias)
torch.cuda.synchronize()

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    record_shapes=True, with_stack=True,
) as prof:
    with torch.profiler.record_function("matmul_add_64x64"):
        matmul_add(a, b, bias)
        torch.cuda.synchronize()

trace_path = os.path.join(TRACE_DIR, "nb_01_warmup_64.json")
prof.export_chrome_trace(trace_path)
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))
perfetto_url(trace_path)

In [ ]:
# --- Large: 4096x4096 (compute-bound) ---
n = 4096
a = torch.randn(n, n, device=DEVICE, dtype=DTYPE)
b = torch.randn(n, n, device=DEVICE, dtype=DTYPE)
bias = torch.randn(n, device=DEVICE, dtype=DTYPE)

for _ in range(5):
    matmul_add(a, b, bias)
torch.cuda.synchronize()

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    record_shapes=True, with_stack=True,
) as prof:
    with torch.profiler.record_function("matmul_add_4096x4096"):
        matmul_add(a, b, bias)
        torch.cuda.synchronize()

trace_path = os.path.join(TRACE_DIR, "nb_01_warmup_4096.json")
prof.export_chrome_trace(trace_path)
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))
perfetto_url(trace_path)

### Key insight

The **same two operations** (matmul + add) look completely different at 64×64 vs 4096×4096.
At small sizes, the GPU is mostly idle between launches — that's **overhead-bound** execution.
At large sizes, one GEMM kernel dominates — that's **compute-bound**.

This is why microbenchmarks at small sizes are misleading.

## Section 2: Real Llama Layer — Full Dispatch Chain

### What to expect

We profile a **single `LlamaDecoderLayer`** from HuggingFace Transformers.
The trace reveals the full dispatch chain of a real transformer layer:

```
RMSNorm → QKV proj → RoPE → Attention (SDPA) → O proj → Residual
       → RMSNorm → Gate/Up proj → SiLU → Down proj → Residual
```

Look for: the GEMMs (QKV, O, gate, up, down) dwarf all pointwise ops.

In [ ]:
from transformers import AutoConfig
from transformers.models.llama.modeling_llama import LlamaDecoderLayer

MODEL_ID = "meta-llama/Llama-3.1-8B"
try:
    config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
except Exception:
    MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    config = AutoConfig.from_pretrained(MODEL_ID)

print(f"Model config : {MODEL_ID}")
print(f"Hidden size  : {config.hidden_size}")
print(f"Num heads    : {config.num_attention_heads}  (KV heads: {config.num_key_value_heads})")
print(f"Intermediate : {config.intermediate_size}")

layer = LlamaDecoderLayer(config, layer_idx=0).to(DEVICE, dtype=DTYPE).eval()

BATCH, SEQ = 4, 512
hidden = torch.randn(BATCH, SEQ, config.hidden_size, device=DEVICE, dtype=DTYPE)
position_ids = torch.arange(SEQ, device=DEVICE).unsqueeze(0).expand(BATCH, -1)

with torch.no_grad():
    for _ in range(5):
        layer(hidden, position_ids=position_ids)
    torch.cuda.synchronize()

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    record_shapes=True, with_stack=True,
) as prof:
    with torch.no_grad():
        with torch.profiler.record_function("llama_decoder_layer"):
            layer(hidden, position_ids=position_ids)
            torch.cuda.synchronize()

trace_path = os.path.join(TRACE_DIR, "nb_02_llama_layer.json")
prof.export_chrome_trace(trace_path)
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=25))

cuda_events = [e for e in prof.key_averages() if e.device_time_total > 0]
print(f"\nTotal CUDA kernels launched: {len(cuda_events)}")
perfetto_url(trace_path)

### Key insight

One decoder layer launches **dozens of CUDA kernels** in eager mode.
The 5 GEMM calls (QKV, O, gate, up, down projections) consume 80%+ of CUDA time.
Everything else — RMSNorm, RoPE, SiLU, residual adds — is pointwise and fast.

This is why kernel fusion (Liger, torch.compile) targets the pointwise ops:
you can't easily beat cuBLAS GEMMs, but you *can* eliminate the overhead between them.

## Section 3: Prefill vs Decode — End-to-End Generation

### What to expect

`model.generate()` has two distinct phases:

1. **Prefill** — processes the entire prompt at once. Large batched GEMMs, compute-bound.
2. **Decode** — generates one token at a time. Tiny sequential GEMMs, memory-bound.

The trace shows: one dense burst of kernels (prefill), then a long sequence of thin repeating groups (decode). Memory profiling reveals KV cache growth.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

GEN_MODEL_ID = "meta-llama/Llama-3.1-8B"
try:
    tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        GEN_MODEL_ID, torch_dtype=DTYPE, device_map=DEVICE, trust_remote_code=True,
    ).eval()
except Exception as e:
    print(f"Could not load {GEN_MODEL_ID}: {e}")
    GEN_MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        GEN_MODEL_ID, torch_dtype=DTYPE, device_map=DEVICE,
    ).eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

prompt = "The key insight about GPU profiling is that"
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    model.generate(**inputs, max_new_tokens=4, do_sample=False)
    torch.cuda.synchronize()

torch.cuda.reset_peak_memory_stats()
mem_before = torch.cuda.memory_allocated()

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    record_shapes=True, profile_memory=True, with_stack=True,
) as prof:
    with torch.no_grad():
        with torch.profiler.record_function("generate"):
            output_ids = model.generate(**inputs, max_new_tokens=32, do_sample=False)

mem_after = torch.cuda.memory_allocated()
mem_peak = torch.cuda.max_memory_allocated()

trace_path = os.path.join(TRACE_DIR, "nb_03_prefill_decode.json")
prof.export_chrome_trace(trace_path)

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=20))

num_prompt = inputs["input_ids"].shape[1]
num_gen = output_ids.shape[1] - num_prompt
generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print(f"\nPrompt tokens    : {num_prompt}")
print(f"Generated tokens : {num_gen}")
print(f"Memory before    : {mem_before / 1024**2:.1f} MB")
print(f"Peak memory      : {mem_peak / 1024**2:.1f} MB")
print(f"KV cache growth  : ~{(mem_after - mem_before) / 1024**2:.1f} MB")
print(f"\nGenerated: {generated}")
perfetto_url(trace_path)

### Key insight

Prefill and decode are **fundamentally different regimes**:
- Prefill is compute-bound: large matrix multiplications, high GPU utilization.
- Decode is memory-bound: loading KV cache from HBM dominates, GPU compute units are underutilized.

This is why optimizations differ: prefill benefits from FlashAttention (less memory movement),
while decode benefits from quantization, speculative decoding, and KV cache compression (MLA).

---
# Part 2: SOTA Kernel Profiling
---

## Section 4: Liger Kernels on Llama

### What to expect

Liger replaces pointwise ops (RMSNorm, SwiGLU, RoPE) with **fused Triton kernels**.
It does NOT touch the GEMMs — those stay as cuBLAS.

We compare:
1. Vanilla `LlamaDecoderLayer` — many small pointwise kernels
2. Liger-patched layer — pointwise clusters collapse into single Triton kernels
3. Bonus: `FusedLinearCrossEntropyLoss` — the crown jewel that never materializes the full logit tensor

In [ ]:
from transformers.models.llama.modeling_llama import LlamaDecoderLayer

BATCH, SEQ = 4, 512
hidden = torch.randn(BATCH, SEQ, config.hidden_size, device=DEVICE, dtype=DTYPE)
position_ids = torch.arange(SEQ, device=DEVICE).unsqueeze(0).expand(BATCH, -1)

# --- Vanilla ---
layer_vanilla = LlamaDecoderLayer(config, layer_idx=0).to(DEVICE, dtype=DTYPE).eval()
with torch.no_grad():
    for _ in range(5):
        layer_vanilla(hidden, position_ids=position_ids)
    torch.cuda.synchronize()

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    record_shapes=True, profile_memory=True, with_stack=True,
) as prof_v:
    with torch.no_grad():
        with torch.profiler.record_function("vanilla_layer"):
            layer_vanilla(hidden, position_ids=position_ids)
            torch.cuda.synchronize()

t_vanilla = sum(e.device_time_total for e in prof_v.key_averages() if e.device_time_total > 0)
k_vanilla = len([e for e in prof_v.key_averages() if e.device_time_total > 0])

prof_v.export_chrome_trace(os.path.join(TRACE_DIR, "nb_04_vanilla.json"))
print("=== Vanilla ===")
print(prof_v.key_averages().table(sort_by="cuda_time_total", row_limit=15))

In [ ]:
# --- Liger-patched ---
try:
    from liger_kernel.transformers import apply_liger_kernel_to_llama
    apply_liger_kernel_to_llama()

    layer_liger = LlamaDecoderLayer(config, layer_idx=0).to(DEVICE, dtype=DTYPE).eval()
    with torch.no_grad():
        for _ in range(5):
            layer_liger(hidden, position_ids=position_ids)
        torch.cuda.synchronize()

    with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
        record_shapes=True, profile_memory=True, with_stack=True,
    ) as prof_l:
        with torch.no_grad():
            with torch.profiler.record_function("liger_layer"):
                layer_liger(hidden, position_ids=position_ids)
                torch.cuda.synchronize()

    t_liger = sum(e.device_time_total for e in prof_l.key_averages() if e.device_time_total > 0)
    k_liger = len([e for e in prof_l.key_averages() if e.device_time_total > 0])

    prof_l.export_chrome_trace(os.path.join(TRACE_DIR, "nb_04_liger.json"))
    print("=== Liger ===")
    print(prof_l.key_averages().table(sort_by="cuda_time_total", row_limit=15))

    speedup = t_vanilla / t_liger if t_liger > 0 else float("inf")
    print(f"\n{'Metric':<25} {'Vanilla':>15} {'Liger':>15}")
    print(f"{'-'*55}")
    print(f"{'CUDA time (us)':<25} {t_vanilla:>15.0f} {t_liger:>15.0f}")
    print(f"{'Kernel launches':<25} {k_vanilla:>15} {k_liger:>15}")
    print(f"{'Speedup':<25} {'':>15} {speedup:>14.2f}x")
except ImportError:
    print("liger-kernel not installed — run: pip install liger-kernel")

In [ ]:
# --- FusedLinearCrossEntropyLoss ---
try:
    from liger_kernel.transformers import LigerFusedLinearCrossEntropyLoss

    vocab_size = 32000
    linear = torch.nn.Linear(config.hidden_size, vocab_size, bias=False, device=DEVICE, dtype=DTYPE)
    ce_loss = torch.nn.CrossEntropyLoss()
    fused_ce = LigerFusedLinearCrossEntropyLoss()

    h_in = torch.randn(BATCH * SEQ, config.hidden_size, device=DEVICE, dtype=DTYPE)
    targets = torch.randint(0, vocab_size, (BATCH * SEQ,), device=DEVICE)

    for _ in range(5):
        logits = linear(h_in)
        ce_loss(logits.float(), targets)
    torch.cuda.synchronize()

    with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
        record_shapes=True,
    ) as prof_ce_std:
        with torch.profiler.record_function("standard_linear_ce"):
            logits = linear(h_in)
            loss = ce_loss(logits.float(), targets)
            loss.backward()
            torch.cuda.synchronize()

    t_std = sum(e.device_time_total for e in prof_ce_std.key_averages() if e.device_time_total > 0)

    linear.zero_grad()
    w_copy = linear.weight.detach().clone().requires_grad_(True)
    h_copy = h_in.detach().clone().requires_grad_(True)

    for _ in range(5):
        fused_ce(h_copy, w_copy, targets)
    torch.cuda.synchronize()

    with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
        record_shapes=True,
    ) as prof_ce_fused:
        with torch.profiler.record_function("fused_linear_ce"):
            loss_f = fused_ce(h_copy, w_copy, targets)
            loss_f.backward()
            torch.cuda.synchronize()

    t_fused = sum(e.device_time_total for e in prof_ce_fused.key_averages() if e.device_time_total > 0)
    ce_speedup = t_std / t_fused if t_fused > 0 else float("inf")

    print(f"Standard Linear+CE : {t_std:.0f} us")
    print(f"Fused Linear CE    : {t_fused:.0f} us")
    print(f"Speedup            : {ce_speedup:.2f}x")
    print(f"\nThe fused version never materializes the [{BATCH*SEQ} x {vocab_size}] logit tensor.")
except ImportError:
    print("LigerFusedLinearCrossEntropyLoss not available")

### Key insight

Liger is a **one-function-call optimization**: `apply_liger_kernel_to_llama()` replaces
pointwise ops with fused Triton kernels.  Fewer kernel launches, less launch overhead.

The crown jewel — `FusedLinearCrossEntropyLoss` — avoids ever materializing the full
`[batch×seq, vocab_size]` logit tensor, saving massive memory and computation during training.

## Section 5: FlashAttention Shootout

### What to expect

Four attention implementations on the **same** Llama-shaped Q/K/V:

| Backend | Kernels | What's happening |
|---------|---------|------------------|
| SDPA math | ~20 | Unfused Q@K, softmax, @V — the reference |
| SDPA flash | 1 | FlashAttention-2, single fused kernel |
| SDPA cuDNN | 1 | cuDNN Hopper-optimized (WGMMA instructions) |
| FA-3 | 1 | Hopper-exclusive: WGMMA + TMA, async softmax pipelining |

In [ ]:
num_heads = config.num_attention_heads
head_dim = config.hidden_size // num_heads
BATCH_ATTN, SEQ_ATTN = 4, 512

q = torch.randn(BATCH_ATTN, num_heads, SEQ_ATTN, head_dim, device=DEVICE, dtype=DTYPE)
k = torch.randn(BATCH_ATTN, num_heads, SEQ_ATTN, head_dim, device=DEVICE, dtype=DTYPE)
v = torch.randn(BATCH_ATTN, num_heads, SEQ_ATTN, head_dim, device=DEVICE, dtype=DTYPE)

print(f"Attention shape: batch={BATCH_ATTN}, heads={num_heads}, seq={SEQ_ATTN}, dim={head_dim}")

results_attn = []

backends = {
    "math": torch.nn.attention.SDPBackend.MATH,
    "flash": torch.nn.attention.SDPBackend.FLASH_ATTENTION,
    "cudnn": torch.nn.attention.SDPBackend.CUDNN_ATTENTION,
}

for name, backend in backends.items():
    try:
        with torch.nn.attention.sdpa_kernel(backend):
            for _ in range(5):
                F.scaled_dot_product_attention(q, k, v, is_causal=True)
            torch.cuda.synchronize()

        with torch.profiler.profile(
            activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
            record_shapes=True, with_stack=True,
        ) as prof_attn:
            with torch.no_grad():
                with torch.profiler.record_function(f"sdpa_{name}"):
                    with torch.nn.attention.sdpa_kernel(backend):
                        F.scaled_dot_product_attention(q, k, v, is_causal=True)
                    torch.cuda.synchronize()

        t = sum(e.device_time_total for e in prof_attn.key_averages() if e.device_time_total > 0)
        nk = len([e for e in prof_attn.key_averages() if e.device_time_total > 0])
        prof_attn.export_chrome_trace(os.path.join(TRACE_DIR, f"nb_05_sdpa_{name}.json"))
        results_attn.append({"name": f"SDPA {name}", "cuda_us": t, "kernels": nk})
        print(f"  [{name}] {t:.0f} us, {nk} kernels")
    except RuntimeError as e:
        print(f"  [{name}] Not available: {e}")

In [ ]:
# --- FlashAttention-3 (Hopper-exclusive) ---
try:
    from flash_attn_interface import flash_attn_func as fa3_func

    q_fa3 = q.transpose(1, 2).contiguous()
    k_fa3 = k.transpose(1, 2).contiguous()
    v_fa3 = v.transpose(1, 2).contiguous()

    for _ in range(5):
        fa3_func(q_fa3, k_fa3, v_fa3, causal=True)
    torch.cuda.synchronize()

    with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
        record_shapes=True, with_stack=True,
    ) as prof_fa3:
        with torch.profiler.record_function("flash_attn_v3"):
            with torch.no_grad():
                fa3_func(q_fa3, k_fa3, v_fa3, causal=True)
            torch.cuda.synchronize()

    t_fa3 = sum(e.device_time_total for e in prof_fa3.key_averages() if e.device_time_total > 0)
    nk_fa3 = len([e for e in prof_fa3.key_averages() if e.device_time_total > 0])
    prof_fa3.export_chrome_trace(os.path.join(TRACE_DIR, "nb_05_fa3.json"))
    results_attn.append({"name": "FlashAttn-3", "cuda_us": t_fa3, "kernels": nk_fa3})
    print(f"  [FA-3] {t_fa3:.0f} us, {nk_fa3} kernels")
except ImportError:
    print("  [FA-3] Not available (build from flash-attention/hopper/)")

In [ ]:
# --- Comparison table ---
if results_attn:
    fastest = min(results_attn, key=lambda r: r["cuda_us"])
    print(f"\n{'Backend':<15} {'CUDA (us)':>12} {'Kernels':>10} {'vs fastest':>12}")
    print(f"{'-'*50}")
    for r in sorted(results_attn, key=lambda r: r["cuda_us"]):
        ratio = r["cuda_us"] / fastest["cuda_us"]
        marker = " <-" if r["name"] == fastest["name"] else ""
        print(f"{r['name']:<15} {r['cuda_us']:>12.0f} {r['kernels']:>10} {ratio:>11.2f}x{marker}")

### Key insight

The math backend launches ~20 kernels for the full Q@K → softmax → @V chain.
All fused backends (flash, cuDNN, FA-3) collapse this to ~1 kernel.

On Hopper, FA-3 uses **WGMMA** (warp group matrix multiply-accumulate) and **TMA**
(Tensor Memory Accelerator) to overlap softmax computation with the next tile's
memory loads — the "async softmax pipelining" that's unique to Hopper.

## Section 6: FP8 on Hopper

### What to expect

Hopper's FP8 Tensor Cores deliver ~2× the TFLOPS of bf16. We compare:
- **bf16 GEMM** via `torch.matmul` (standard cuBLAS path)
- **FP8 GEMM** via `torch._scaled_mm` with `float8_e4m3fn`

Dimensions are from the Llama model's gate/up projection: `[batch*seq, hidden] @ [hidden, intermediate]`.
Look for different kernel names in the traces — FP8 invokes different Tensor Core instructions.

In [ ]:
M = BATCH * SEQ
K = config.hidden_size
N = config.intermediate_size

print(f"GEMM dimensions: [{M} x {K}] @ [{K} x {N}]")
print(f"(Simulates gate/up projection: hidden -> intermediate)\n")

a_bf16 = torch.randn(M, K, device=DEVICE, dtype=torch.bfloat16)
b_bf16 = torch.randn(K, N, device=DEVICE, dtype=torch.bfloat16)

for _ in range(5):
    torch.matmul(a_bf16, b_bf16)
torch.cuda.synchronize()

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    record_shapes=True,
) as prof_bf16:
    with torch.profiler.record_function("gemm_bf16"):
        torch.matmul(a_bf16, b_bf16)
        torch.cuda.synchronize()

t_bf16 = sum(e.device_time_total for e in prof_bf16.key_averages() if e.device_time_total > 0)
prof_bf16.export_chrome_trace(os.path.join(TRACE_DIR, "nb_06_gemm_bf16.json"))

fp8_max = torch.finfo(torch.float8_e4m3fn).max
a_fp8 = a_bf16.clamp(-fp8_max, fp8_max).to(torch.float8_e4m3fn)
b_fp8 = b_bf16.clamp(-fp8_max, fp8_max).to(torch.float8_e4m3fn)
scale_a = torch.ones(1, device=DEVICE, dtype=torch.float32)
scale_b = torch.ones(1, device=DEVICE, dtype=torch.float32)

for _ in range(5):
    torch._scaled_mm(a_fp8, b_fp8.t(), scale_a=scale_a, scale_b=scale_b, out_dtype=torch.bfloat16)
torch.cuda.synchronize()

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    record_shapes=True,
) as prof_fp8:
    with torch.profiler.record_function("gemm_fp8"):
        torch._scaled_mm(a_fp8, b_fp8.t(), scale_a=scale_a, scale_b=scale_b, out_dtype=torch.bfloat16)
        torch.cuda.synchronize()

t_fp8 = sum(e.device_time_total for e in prof_fp8.key_averages() if e.device_time_total > 0)
prof_fp8.export_chrome_trace(os.path.join(TRACE_DIR, "nb_06_gemm_fp8.json"))

speedup_fp8 = t_bf16 / t_fp8 if t_fp8 > 0 else float("inf")

print(f"bf16 GEMM : {t_bf16:.0f} us")
print(f"FP8  GEMM : {t_fp8:.0f} us")
print(f"Speedup   : {speedup_fp8:.2f}x")

### Key insight

FP8 uses different Tensor Core instructions than bf16 — you can see this in the kernel names.
The speedup comes from:
1. **2× throughput** from the FP8 Tensor Core data path on Hopper
2. **Half the memory bandwidth** needed to load operands (8 bits vs 16 bits)

`torch._scaled_mm` is the low-level API.  Production frameworks (TransformerEngine, FBGEMM)
add per-tensor/per-channel scaling factors for accuracy.

## Section 7: FlashMLA — DeepSeek's Flagship Kernel

### What to expect

MLA (Multi-head Latent Attention) from DeepSeek-V2 compresses KV into a low-rank latent
space.  FlashMLA is the decode-attention kernel optimized for this compressed layout.

**Seesaw scheduling:** Unlike split-KV (FlashDecoding) where work is evenly divided and
some SMs finish early and idle, FlashMLA's tile scheduler dynamically assigns variable-length
tile groups to SMs, avoiding the "tail effect."

We profile:
1. `get_mla_metadata` — the tile scheduler setup
2. `flash_mla_with_kvcache` — the actual paged decode kernel

In [ ]:
try:
    from flash_mla import get_mla_metadata, flash_mla_with_kvcache

    batch_mla = 4
    num_heads_mla = 16
    head_dim_v = 128
    max_seqlen = 2048
    page_size = 64

    num_pages_per_seq = (max_seqlen + page_size - 1) // page_size
    total_pages = num_pages_per_seq * batch_mla

    print(f"FlashMLA config: batch={batch_mla}, heads={num_heads_mla}, "
          f"head_dim={head_dim_v}, seqlen={max_seqlen}, page_size={page_size}")

    cache_seqlens = torch.full((batch_mla,), max_seqlen, device=DEVICE, dtype=torch.int32)

    tile_scheduler_metadata, num_splits = get_mla_metadata(
        cache_seqlens, num_heads_mla * head_dim_v, num_heads_mla
    )
    print(f"num_splits: {num_splits}")

    kv_cache = torch.randn(
        total_pages, page_size, 2, num_heads_mla, head_dim_v,
        device=DEVICE, dtype=DTYPE,
    )
    block_table = torch.arange(
        total_pages, device=DEVICE, dtype=torch.int32,
    ).reshape(batch_mla, num_pages_per_seq)

    q_mla = torch.randn(batch_mla, num_heads_mla, head_dim_v, device=DEVICE, dtype=DTYPE)

    for _ in range(5):
        flash_mla_with_kvcache(
            q_mla, kv_cache, block_table, cache_seqlens,
            head_dim_v, tile_scheduler_metadata, num_splits,
            softmax_scale=head_dim_v ** -0.5,
        )
    torch.cuda.synchronize()

    with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
        record_shapes=True, with_stack=True,
    ) as prof_mla:
        with torch.profiler.record_function("flash_mla_decode"):
            with torch.no_grad():
                out_mla, lse_mla = flash_mla_with_kvcache(
                    q_mla, kv_cache, block_table, cache_seqlens,
                    head_dim_v, tile_scheduler_metadata, num_splits,
                    softmax_scale=head_dim_v ** -0.5,
                )
            torch.cuda.synchronize()

    trace_path = os.path.join(TRACE_DIR, "nb_07_flashmla.json")
    prof_mla.export_chrome_trace(trace_path)

    t_mla = sum(e.device_time_total for e in prof_mla.key_averages() if e.device_time_total > 0)
    kv_bytes = kv_cache.nelement() * kv_cache.element_size()
    bw = kv_bytes / (t_mla * 1e-6) / 1e9 if t_mla > 0 else 0

    print(prof_mla.key_averages().table(sort_by="cuda_time_total", row_limit=10))
    print(f"\nCUDA time     : {t_mla:.0f} us")
    print(f"KV cache      : {kv_bytes / 1024**2:.1f} MB")
    print(f"Effective BW  : {bw:.1f} GB/s")
    print(f"Output shape  : {out_mla.shape}")
    perfetto_url(trace_path)

except ImportError:
    print("FlashMLA not installed — build from source (outside FlashMLA-src/ directory)")

### Key insight

FlashMLA achieves near-peak HBM bandwidth for decode attention by:
1. **Paged KV cache** — no fragmentation, no wasted memory on padding
2. **Seesaw tile scheduling** — dynamically balances work across SMs
3. **Hopper-specific instructions** — WGMMA for compute, TMA for async loads

The `get_mla_metadata` call is cheap (runs on CPU/CUDA) but encodes the entire
scheduling strategy that makes the decode kernel efficient.

## Section 8: torch.compile Modes

### What to expect

Three compilation modes on the **same** LlamaDecoderLayer:

| Mode | What it does | Trade-off |
|------|-------------|----------|
| `default` | Fuses pointwise ops, fewer kernel launches | Fast compile, moderate speedup |
| `reduce-overhead` | Wraps in CUDA graphs — one replay kernel | Medium compile, minimal launch overhead |
| `max-autotune` | Benchmarks multiple kernel impls, picks fastest | Slow compile, best steady-state |

In [ ]:
import time

BATCH_C, SEQ_C = 4, 512
hidden_c = torch.randn(BATCH_C, SEQ_C, config.hidden_size, device=DEVICE, dtype=DTYPE)
pos_ids_c = torch.arange(SEQ_C, device=DEVICE).unsqueeze(0).expand(BATCH_C, -1)

# Eager baseline
layer_eager = LlamaDecoderLayer(config, layer_idx=0).to(DEVICE, dtype=DTYPE).eval()
with torch.no_grad():
    for _ in range(5):
        layer_eager(hidden_c, position_ids=pos_ids_c)
    torch.cuda.synchronize()

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    record_shapes=True, with_stack=True,
) as prof_eager:
    with torch.no_grad():
        with torch.profiler.record_function("eager_baseline"):
            layer_eager(hidden_c, position_ids=pos_ids_c)
            torch.cuda.synchronize()

eager_us = sum(e.device_time_total for e in prof_eager.key_averages() if e.device_time_total > 0)
prof_eager.export_chrome_trace(os.path.join(TRACE_DIR, "nb_08_eager.json"))

compile_results = []
for mode in ["default", "reduce-overhead", "max-autotune"]:
    fresh = LlamaDecoderLayer(config, layer_idx=0).to(DEVICE, dtype=DTYPE).eval()
    compiled = torch.compile(fresh, mode=mode)

    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(5):
            compiled(hidden_c, position_ids=pos_ids_c)
        torch.cuda.synchronize()
    ct = time.perf_counter() - t0

    with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
        record_shapes=True, with_stack=True,
    ) as prof_c:
        with torch.no_grad():
            with torch.profiler.record_function(f"compile_{mode}"):
                compiled(hidden_c, position_ids=pos_ids_c)
                torch.cuda.synchronize()

    cuda_us = sum(e.device_time_total for e in prof_c.key_averages() if e.device_time_total > 0)
    prof_c.export_chrome_trace(os.path.join(TRACE_DIR, f"nb_08_compile_{mode}.json"))
    compile_results.append({"mode": mode, "compile_s": ct, "cuda_us": cuda_us})

print(f"{'Mode':<20} {'Compile (s)':>12} {'CUDA (us)':>12} {'Speedup':>10}")
print(f"{'-'*55}")
print(f"{'eager':<20} {'-':>12} {eager_us:>12.0f} {'1.00x':>10}")
for r in compile_results:
    sp = eager_us / r["cuda_us"] if r["cuda_us"] > 0 else float("inf")
    print(f"{r['mode']:<20} {r['compile_s']:>12.2f} {r['cuda_us']:>12.0f} {sp:>9.2f}x")

### Key insight

- `default` fuses pointwise ops and reduces kernel count with minimal compile time.
- `reduce-overhead` wraps execution in CUDA graphs — almost zero launch overhead, but
  requires static shapes and disallows Python-side control flow during execution.
- `max-autotune` tries multiple kernel implementations and picks the fastest for each op.
  Longest compile time, but best steady-state performance.

In production: use `reduce-overhead` for serving (fixed shapes), `max-autotune` for training.

---
# Part 3: Nsight Deep Dive
---

## Section 9: Nsight Systems

### What to expect

`torch.profiler` captures what's happening **from Python's perspective**.
Nsight Systems captures what's happening **from the GPU's perspective** — with hardware-level
accuracy, correlating CPU API calls, CUDA launches, memory transfers, and GPU execution.

We use **NVTX** (NVIDIA Tools Extension) to annotate code regions so they appear as
labeled ranges in the Nsight timeline.

In [ ]:
# NVTX annotation demo — this cell shows the API, not a full nsys run
layer_nsight = LlamaDecoderLayer(config, layer_idx=0).to(DEVICE, dtype=DTYPE).eval()
hidden_ns = torch.randn(4, 512, config.hidden_size, device=DEVICE, dtype=DTYPE)
pos_ns = torch.arange(512, device=DEVICE).unsqueeze(0).expand(4, -1)

with torch.no_grad():
    for _ in range(3):
        layer_nsight(hidden_ns, position_ids=pos_ns)
    torch.cuda.synchronize()

torch.cuda.nvtx.range_push("prefill")
with torch.no_grad():
    layer_nsight(hidden_ns, position_ids=pos_ns)
torch.cuda.synchronize()
torch.cuda.nvtx.range_pop()

for step in range(4):
    decode_h = torch.randn(4, 1, config.hidden_size, device=DEVICE, dtype=DTYPE)
    decode_p = torch.full((4, 1), 512 + step, device=DEVICE, dtype=torch.long)

    torch.cuda.nvtx.range_push(f"decode_step_{step}")
    with torch.no_grad():
        layer_nsight(decode_h, position_ids=decode_p)
    torch.cuda.synchronize()
    torch.cuda.nvtx.range_pop()

print("NVTX regions pushed: prefill, decode_step_0..3")
print("Run with nsys to capture these in a timeline.")

### Nsight Systems commands

Run from the terminal (not inside the notebook):

```bash
# Full timeline with NVTX annotations
nsys profile \
  --stats=true \
  --output=traces/09_nsight_layer \
  --trace=cuda,nvtx \
  --force-overwrite=true \
  ./scripts/09_nsight_target.py --workload layer

# Just attention
nsys profile \
  --stats=true \
  --output=traces/09_nsight_attn \
  --trace=cuda,nvtx \
  --force-overwrite=true \
  ./scripts/09_nsight_target.py --workload attention
```

Open `.nsys-rep` files in **Nsight Systems UI** (not Perfetto).
Look for NVTX rows showing `prefill` vs `decode_step_N` with GPU kernels aligned underneath.

### Key insight

Nsight Systems answers: "Where is the GPU idle? Where is it waiting for CPU? Where is data moving?"
torch.profiler answers: "Which ops took the most time?"

Use torch.profiler for quick iteration, Nsight Systems for deep investigation.

## Section 10: Nsight Compute + Roofline

### What to expect

Nsight Compute profiles a **single kernel** at hardware-counter granularity:
occupancy, memory throughput, compute throughput, stall reasons.

The **roofline model** plots achieved TFLOPS vs arithmetic intensity:
- Kernels below the memory roof are **memory-bound** (need to reduce data movement)
- Kernels below the compute roof are **compute-bound** (need to increase parallelism)
- Kernels in the "attainable" region are well-optimized

### Nsight Compute commands

```bash
# Profile the top 5 GEMM kernels during prefill
ncu \
  --set full \
  --output=traces/09_ncu_layer \
  --nvtx \
  --nvtx-include "prefill/" \
  --kernel-name regex:gemm \
  --launch-count 5 \
  ./scripts/09_nsight_target.py --workload layer

# Profile all kernels in attention (for roofline)
ncu \
  --set full \
  --output=traces/09_ncu_attn \
  --nvtx \
  --nvtx-include "self_attention_forward/" \
  ./scripts/09_nsight_target.py --workload attention
```

### Reading the roofline

| Metric | What it tells you |
|--------|------------------|
| Arithmetic Intensity (FLOP/byte) | How much compute per byte loaded from memory |
| Peak TFLOPS (roofline ceiling) | H200: ~989 TFLOPS bf16, ~1979 TFLOPS FP8 |
| Peak Bandwidth (memory roof) | H200: 4.8 TB/s HBM3e |
| Achieved TFLOPS | Where your kernel actually lands |

If a GEMM achieves 80%+ of peak TFLOPS for its arithmetic intensity → it's well-optimized.
If FlashAttention achieves 80%+ of peak bandwidth → it's memory-bandwidth limited (by design).

### Key insight

Nsight Compute is the only tool that can tell you **why** a kernel is slow:
- Low occupancy? Not enough threads to hide latency.
- Low memory throughput? Uncoalesced accesses or bank conflicts.
- Low compute throughput? Pipeline stalls or branch divergence.

Use `--kernel-name regex:<pattern>` and `--nvtx-include` to target specific kernels — profiling everything is very slow.

---
# Appendix
---

## Section 11: Cheatsheet

### Profiler table patterns

```python
# Basic profile
with torch.profiler.profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
) as prof:
    model(x)
    torch.cuda.synchronize()  # ALWAYS synchronize inside the profiler!

# Print table sorted by CUDA time
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=20))

# With memory tracking
with torch.profiler.profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    profile_memory=True,
) as prof:
    ...

# Export Chrome trace for Perfetto
prof.export_chrome_trace("trace.json")
```

### Kernel name decoding

| Pattern | What it is |
|---------|------------|
| `ampere_bf16_...gemm` | cuBLAS GEMM on Ampere (bf16) |
| `sm90_xmma_gemm_...` | cuBLAS GEMM on Hopper (WGMMA instructions) |
| `void cutlass::...` | CUTLASS template-based GEMM |
| `triton_...` | Triton-compiled kernel (torch.compile or Liger) |
| `void flash::...fwd_kernel` | FlashAttention forward kernel |
| `cudnn::...` | cuDNN attention kernel |
| `at::native::vectorized_elementwise_kernel` | PyTorch's generic pointwise kernel |
| `void at::native::reduce_kernel` | Reduction (sum, mean, norm) |

### Tool selection

| Question | Tool |
|----------|------|
| "Which ops are slowest?" | `torch.profiler` table |
| "What does the timeline look like?" | `torch.profiler` → Perfetto |
| "Where is the GPU idle?" | Nsight Systems |
| "WHY is this kernel slow?" | Nsight Compute |
| "Is this memory- or compute-bound?" | Nsight Compute roofline |
| "How much VRAM does each op use?" | `torch.profiler` with `profile_memory=True` |

## Section 12: SOTA Kernel Landscape

A brief survey of the cutting-edge kernels beyond what we profiled today.

### What we profiled

| Kernel | Origin | Key technique | Profiled in |
|--------|--------|--------------|-------------|
| FlashAttention-2 | Dao-AILab | Tiling + online softmax | Section 5 (SDPA flash) |
| FlashAttention-3 | Dao-AILab | WGMMA + TMA + async softmax | Section 5 |
| Liger Kernel | LinkedIn | Fused Triton for pointwise ops | Section 4 |
| FlashMLA | DeepSeek | Seesaw scheduling for MLA decode | Section 7 |
| FP8 Tensor Cores | NVIDIA/PyTorch | float8_e4m3fn via _scaled_mm | Section 6 |

### Worth knowing about

| Kernel | What it is | Status |
|--------|-----------|--------|
| **ThunderKittens** (HazyResearch) | CUDA C++ DSL for tile-level programming on Hopper. Builds kernels from register/shared-memory tile primitives. Achieves near-peak TFLOPS. | Research, Hopper-only |
| **DeepGEMM** (DeepSeek) | JIT-compiled FP8 GEMMs for MoE. Groups experts into batched GEMMs, uses DeepSeek's custom tiling. | Production at DeepSeek |
| **Mega MoE** (DeepSeek) | Communication-overlapped MoE execution across nodes. Overlaps all-to-all with expert computation. | Cluster-scale, not single-GPU |
| **CUTLASS 3.x** (NVIDIA) | Template-based GEMM library using CuTe layout algebra. Foundation for many fused kernels. | Production |
| **TransformerEngine** (NVIDIA) | FP8 training/inference with automatic scaling, delayed scaling, and mixed-precision recipes. | Production |

### The trend

Every generation pushes more work into fewer, larger kernel launches:
- **2022:** FlashAttention showed that fusing Q@K+softmax+@V into one kernel saves 10× memory
- **2024:** FA-3 showed that Hopper's async primitives (TMA, WGMMA) enable overlapping compute with memory loads *within* a kernel
- **2025:** FlashMLA and DeepGEMM showed that scheduling strategy (how work is distributed across SMs) matters as much as the math

**References:**
- [FlashAttention-3 Paper](https://arxiv.org/abs/2407.08608)
- [Liger Kernel Paper](https://arxiv.org/abs/2410.10989)
- [FlashMLA Repository](https://github.com/deepseek-ai/FlashMLA)
- [ThunderKittens Repository](https://github.com/HazyResearch/ThunderKittens)
- [DeepGEMM Repository](https://github.com/deepseek-ai/DeepGEMM)
- [PyTorch Profiler Docs](https://pytorch.org/docs/stable/profiler.html)
- [Nsight Systems User Guide](https://docs.nvidia.com/nsight-systems/)
- [Nsight Compute User Guide](https://docs.nvidia.com/nsight-compute/)